# Teste do JAGC (ResNet-18) nos folds do dataset combinado, só com imagens Acne04

Mesma receita exata do JAGC (ImageNet + BatchNorm congelado, SGD, 120 épocas fixas, imagens 300px), nos folds do dataset combinado Acne04+AcneLevel, mas **usando apenas as imagens Acne04** (sem as cópias tratadas do AcneLevel).

**Antes de começar:** suba o arquivo `jagc_colab_bundle.zip` para a raiz do seu Google Drive (`Meu Drive`) e escolha *Ambiente de execução > Alterar tipo de ambiente > GPU (T4)*.

Os resultados são salvos no seu Drive em `jagc_results/`. O script pula os folds já concluídos, então se a sessão cair basta rodar as células de novo.

In [ ]:
# 1) GPU e Drive
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Usa o torch que o Colab já traz (não instalar outra versão). Só garante as bibliotecas auxiliares.
!pip -q install scikit-learn pillow
import torch, torchvision
print(torch.__version__, torchvision.__version__, "GPU:", torch.cuda.is_available())


In [ ]:
# 3) Descompacta o pacote (código + splits + 2863 imagens 300px) no disco local do Colab (mais rápido que o Drive)
!rm -rf /content/pibic_final
!unzip -q /content/drive/MyDrive/jagc_colab_bundle.zip -d /content
!ls /content/pibic_final /content/pibic_final/data/jagc_replica_300_combined | head -5
!ls /content/pibic_final/data/jagc_replica_300_combined | wc -l

In [ ]:
# 4) Teste rápido (opcional, ~3 min): 10 épocas só no fold 0, para validar o ambiente. Resultado vai para uma pasta de teste.
%cd /content/pibic_final/training
!python run_jagc_original_replica.py --fold 0 --arch resnet18 --epochs 10 --name smoke_colab \
    --splits results/kfold_splits_wholeimage_v2.json --manifest-v2 ../data/final_wholeimage_v2/manifest.json \
    --img-dir ../data/jagc_replica_300_combined --only-source a04 --num-workers 2 --output-dir /content/smoke_out 2>&1 | grep -v -i "warn\|F.softmax"

In [ ]:
# 5) EXECUÇÃO COMPLETA: 5 folds, seed 42. Resultados persistem no Drive; folds prontos são pulados.
%cd /content/pibic_final/training
OUT = "/content/drive/MyDrive/jagc_results"
SEED = 42
name = "jagc_replica_combined_a04only" if SEED == 42 else f"jagc_replica_combined_a04only_seed{SEED}"
for fold in range(5):
    !python run_jagc_original_replica.py --fold {fold} --arch resnet18 --seed {SEED} --name {name} \
        --splits results/kfold_splits_wholeimage_v2.json --manifest-v2 ../data/final_wholeimage_v2/manifest.json \
        --img-dir ../data/jagc_replica_300_combined --only-source a04 --num-workers 2 --output-dir {OUT} 2>&1 | grep -v -i "warn\|F.softmax"

In [ ]:
# 6) Métricas agregadas (5 folds juntos)
import numpy as np, glob
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix
d = f"/content/drive/MyDrive/jagc_results/{name}"
fs = sorted(glob.glob(d + "/fold*.npz"))
P = np.concatenate([np.load(f, allow_pickle=True)["probs"] for f in fs])
Y = np.concatenate([np.load(f, allow_pickle=True)["labels"] for f in fs])
pr = P.argmax(1)
conf, ok = P.max(1), (pr == Y)
ece = sum((((conf > i/10) & (conf <= (i+1)/10)).mean()) * abs(ok[(conf > i/10) & (conf <= (i+1)/10)].mean() - conf[(conf > i/10) & (conf <= (i+1)/10)].mean())
          for i in range(10) if ((conf > i/10) & (conf <= (i+1)/10)).any())
print(f"folds concluidos: {len(fs)}/5  n={len(Y)}")
print(f"Acc={accuracy_score(Y,pr)*100:.2f}%  macro-F1={f1_score(Y,pr,average='macro')*100:.2f}%  QWK={cohen_kappa_score(Y,pr,weights='quadratic'):.3f}  ECE={ece:.4f}")
print(confusion_matrix(Y, pr))

Ao terminar, a pasta `jagc_results/jagc_replica_combined_a04only/` no seu Drive contém um `.npz` por fold e os históricos por época. Me avise (ou envie a pasta) que eu agrego e comparo com os resultados anteriores:

- receita do JAGC nos folds combinados com as duas versões das imagens: 76,7% (ensemble de 3 seeds)
- receita do JAGC no split oficial: ~80,8%